# AI-for-QEC paper workflow

这是论文实验的编排源码模板。当前仓库只有 contracts，没有 Stim、CUDA、训练、MWPM、恢复或性能评估实现。

执行 `Run All` 只会验证导入、配置和函数定义；只有显式传入满足 `qec.NotebookPlatform` 的 runtime 后，`run_paper(runtime)` 才会启动实验。

In [ ]:
from __future__ import annotations

import ai_qec.notebook_api as qec

## Experiment configuration

配置保持普通 Python 数据结构，后续 runtime 必须在创建 Run 或数据之前完成 schema 与资源验证。

In [ ]:
CONFIG: dict[str, object] = {
    "schema_version": "0.1-contract",
    "qec": {
        "code_family": "surface",
        "distance": 3,
        "rounds": 3,
        "logical_basis": "Z",
        "circuit_family": "memory",
    },
    "noise": {
        "family": "circuit-level-depolarizing",
        "parameters": {"physical_error_rate": 0.001},
    },
    "dataset": {
        "generator": "stim",
        "generator_version": "unresolved",
        "backend_semantics": "exact",
        "train_samples": 10000,
        "validation_samples": 2000,
        "test_samples": 5000,
        "seed": 42,
        "split_policy": "fixed",
        "schema_version": "qec-batch-v1",
    },
    "model": {
        "family": "unresolved",
        "architecture_version": "unresolved",
    },
    "training": {
        "optimizer": "adam",
        "learning_rate": 0.001,
        "epochs": 10,
        "batch_size": 256,
        "scheduler": "none",
        "loss": "unresolved",
    },
    "execution": {
        "device": "cuda",
        "cpu_count": 8,
        "gpu_count": 1,
        "distributed": False,
        "num_workers": 4,
        "mixed_precision": True,
        "compile_model": False,
    },
    "scientific_evaluation": {
        "baseline_decoders": ["mwpm"],
        "primary_metric": "logical_error_rate",
        "confidence_level": 0.95,
        "stopping_rule": "fixed-shots",
        "invalid_sample_policy": "fail",
    },
    "accuracy_gate": {
        "gate_id": "gate-a",
        "baseline_decoder": "mwpm",
        "primary_metric": "logical_error_rate",
        "comparison_rule": "non-inferiority",
        "tolerance": 0.0,
        "confidence_level": 0.95,
    },
}

## Standard orchestration

Accuracy Gate 是独立判定对象。只有 Gate PASS 才调用性能评估；Gate FAIL 仍然执行可视化和收尾。

In [ ]:
def run_paper(
    runtime: qec.NotebookPlatform,
) -> tuple[
    qec.ScientificEvaluationResult,
    qec.ScientificAcceptanceResult,
    qec.ArtifactRef | None,
    tuple[qec.ArtifactRef, ...],
]:
    experiment = runtime.create_experiment(CONFIG)
    run = experiment.start_or_recover()

    dataset = run.resolve_dataset()
    model = run.train(dataset)

    scientific_result = run.evaluate_accuracy(
        model=model,
        dataset=dataset,
        baselines=("mwpm",),
    )
    acceptance = run.check_accuracy_gate(scientific_result)

    performance_result = None
    if acceptance.decision is qec.GateDecision.PASS:
        performance_result = run.evaluate_performance(
            model=model,
            dataset=dataset,
        )

    figures = run.visualize(
        scientific_result=scientific_result,
        acceptance=acceptance,
        performance_result=performance_result,
    )
    run.finish()
    return scientific_result, acceptance, performance_result, figures

## Runtime injection

后续实现完成后，由调用环境显式设置 runtime 并调用 `run_paper(runtime)`。当前不执行研究流程。

In [ ]:
RUNTIME: qec.NotebookPlatform | None = None
assert RUNTIME is None
print("Contract-only notebook loaded; no experiment was executed.")